# HDFS_v1：資料載入與初步探索

這份 Notebook 先做最基本、也最重要的資料健檢。目標是把 HDFS block-level event occurrence matrix 整理成後續模型會使用的 `X`（事件計數）與 `y`（normal/anomaly label）。

本階段不修改原始 CSV，也不訓練複雜模型。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 40)
plt.style.use('seaborn-v0_8-whitegrid')

# 同時支援從 repo 根目錄或 0722_experiment 目錄啟動 Notebook。
candidates = [
    Path('data/HDFS_v1/preprocessed'),
    Path('0722_experiment/data/HDFS_v1/preprocessed'),
]
DATA_DIR = next((p for p in candidates if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError('找不到 HDFS_v1/preprocessed；請從 repo 根目錄或 0722_experiment 啟動。')

MATRIX_PATH = DATA_DIR / 'Event_occurrence_matrix.csv'
LABEL_PATH = DATA_DIR / 'anomaly_label.csv'
assert MATRIX_PATH.exists() and LABEL_PATH.exists(), '缺少必要的 preprocessing CSV。'
print(f'Data directory: {DATA_DIR.resolve()}')

## 1. 載入兩份核心資料

`Event_occurrence_matrix.csv` 的每一列是一個 block trace，`E1` 到 `E29` 是各 event template 的出現次數。`anomaly_label.csv` 則提供每個 `BlockId` 的正式 Normal/Anomaly 標籤。

In [ ]:
matrix_df = pd.read_csv(MATRIX_PATH)
labels_df = pd.read_csv(LABEL_PATH)
event_cols = [c for c in matrix_df.columns if c.startswith('E') and c[1:].isdigit()]

print('Event occurrence matrix:', matrix_df.shape)
print('Anomaly labels:', labels_df.shape)
print('Number of event features:', len(event_cols))
display(matrix_df.head())
display(labels_df.head())

## 2. 對齊與品質檢查

不能只假設兩個 CSV 的列順序相同，因此使用 `BlockId` 做 one-to-one merge。以下同時檢查重複 ID、缺失值、負的 event count，以及 matrix 內的 Success/Fail 是否和正式 label 一致。

In [ ]:
assert not matrix_df['BlockId'].duplicated().any(), 'Matrix 中有重複 BlockId。'
assert not labels_df['BlockId'].duplicated().any(), 'Label file 中有重複 BlockId。'

data = matrix_df.merge(
    labels_df.rename(columns={'Label': 'GroundTruth'}),
    on='BlockId', how='inner', validate='one_to_one'
)
assert len(data) == len(matrix_df) == len(labels_df), '兩份資料的 BlockId 未完全對齊。'
assert data[event_cols].isna().sum().sum() == 0, 'Event matrix 含缺失值。'
assert (data[event_cols] >= 0).all().all(), 'Event count 不應為負數。'

expected = data['Label'].map({'Success': 'Normal', 'Fail': 'Anomaly'})
consistency_rate = expected.eq(data['GroundTruth']).mean()
constant_events = [c for c in event_cols if data[c].nunique(dropna=False) <= 1]
zero_fraction = (data[event_cols] == 0).to_numpy().mean()

quality_summary = pd.Series({
    'trace_count': len(data),
    'event_feature_count': len(event_cols),
    'duplicate_block_ids': data['BlockId'].duplicated().sum(),
    'missing_event_values': data[event_cols].isna().sum().sum(),
    'label_consistency_rate': consistency_rate,
    'zero_fraction': zero_fraction,
})
display(quality_summary.to_frame('value'))
print('Constant event columns:', constant_events or 'None')

## 3. Label 分布

異常偵測資料通常高度不平衡，因此只看 accuracy 很容易產生誤導。後續應優先觀察 precision、recall、F1、PR-AUC 與 ROC-AUC。

In [ ]:
label_counts = data['GroundTruth'].value_counts().reindex(['Normal', 'Anomaly'])
label_percent = label_counts / label_counts.sum() * 100
display(pd.DataFrame({'count': label_counts, 'percent': label_percent.round(2)}))

ax = label_counts.plot.bar(color=['#4C78A8', '#E45756'], figsize=(6, 4))
ax.set(title='HDFS_v1 label distribution', xlabel='', ylabel='Number of block traces')
ax.tick_params(axis='x', rotation=0)
for container in ax.containers:
    ax.bar_label(container, fmt='%d', padding=3)
plt.tight_layout()
plt.show()

## 4. Event counts 的簡單探索

先計算每個 trace 的事件總數，並比較 Normal/Anomaly。直方圖使用 `log1p(total events)`，避免少數非常長的 trace 把大多數資料擠在圖的左側。

In [ ]:
data['TotalEvents'] = data[event_cols].sum(axis=1)
display(data.groupby('GroundTruth')['TotalEvents'].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for label, color in [('Normal', '#4C78A8'), ('Anomaly', '#E45756')]:
    values = np.log1p(data.loc[data['GroundTruth'] == label, 'TotalEvents'])
    axes[0].hist(values, bins=50, alpha=0.55, density=True, label=label, color=color)
axes[0].set(title='Trace size distribution', xlabel='log(1 + total event count)', ylabel='Density')
axes[0].legend()

mean_counts = data.groupby('GroundTruth')[event_cols].mean().T
mean_counts['Overall'] = data[event_cols].mean()
top_events = mean_counts.nlargest(10, 'Overall').index
mean_counts.loc[top_events, ['Normal', 'Anomaly']].plot.bar(ax=axes[1], color=['#4C78A8', '#E45756'])
axes[1].set(title='Top 10 events: mean count per trace', xlabel='Event template', ylabel='Mean count')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 5. 建立模型輸入 `X` 和 `y`

`BlockId`、`Label`、`Type` 都不是 event-count 特徵，所以不放入 `X`。正式 ground truth 轉成 Normal=0、Anomaly=1。常數欄不含區分資訊，這裡先列出並排除。

注意：這裡尚未切分資料或 scaling。後續必須先切 train/validation/test，再只使用 training set fit preprocessing，才能避免資料洩漏。

In [ ]:
usable_event_cols = [c for c in event_cols if c not in constant_events]
X = data[usable_event_cols].copy()
y = data['GroundTruth'].map({'Normal': 0, 'Anomaly': 1}).astype('int8')
block_ids = data['BlockId'].copy()

print('X shape:', X.shape)
print('y shape:', y.shape)
print('Features removed because they are constant:', constant_events)
print(f'Anomaly rate: {y.mean():.2%}')
display(X.head())

## 初步結論與下一步

完成本 Notebook 後，我們已經有乾淨的 `X`、`y` 與 `block_ids`。下一版可以加入分層 train/validation/test split，並比較 raw counts、`log1p` 與 scaling。若使用 autoencoder 做異常偵測，通常只以 training split 中的 Normal traces 訓練，再用 validation set 選 reconstruction-error threshold，最後只在 test set 報告一次結果。